# B-MiRA v2.0 — workflow notebook

This notebook only **configures, runs and inspects** the agent. All logic lives in the `bmira/` package next to it.

| Module | Role |
|---|---|
| `config.py` | every tunable value (`Settings`) |
| `schemas.py` | vocabularies and data models |
| `llm.py` | prompts + LangChain client (counts every call) |
| `sources.py` | PubMed / Europe PMC (PublicationType, retractions, query translation) |
| `normalize.py` | entity resolver with alias registry, relation lexicon, span anchoring |
| `evidence.py` | claim grading, negation-aware verb check |
| `semantic.py` | embeddings → LLM pair verdicts → complete-linkage families → conflicts |
| `portfolio.py` | **pathway portfolio**: link evidence, logic checks, scoring, budget allocation |
| `graph.py` | LangGraph nodes, routing, report |
| `chat.py` | follow-up questions answered only from a finished run |
| `offline.py` | surrogate LLM + fixture corpus + hashing encoder for key-free runs |

For a chat interface instead of this notebook: `streamlit run app.py`.

Run order: §1 setup → §2 offline run (no keys, no network) → §3 inspect → §4 tests → §5 live run.

## §1. Setup

In [1]:
# Colab: upload/clone the project folder first, then run from its root.
# !pip -q install -r requirements.txt
import sys, json
sys.path.insert(0, ".")
import pandas as pd
from bmira import Settings, Runtime, build_agent, run
from bmira.offline import offline_runtime
pd.set_option("display.max_colwidth", 80)

## §2. Offline run on a synthetic scenario

The scenario (`bmira/fixtures/lactate_cd8.json`) is **invented** to exercise the loop, not to make biological claims. It is built so the LLM's favourite pathway (chromatin route) is wrong and the best-supported route (redox-metabolic) is *not* among the seeds. A correct run must still finish with the redox route leading.

Every pathway and every step gets one of three verdicts: **Supported**, **Contradicted**, or **Insufficient evidence** (with a reason).

What to watch in the log:
- round 1: retracted paper excluded, stitched span dropped, alias merges (lactic acid → lactate), 3 seeded pathways plus 1 gated expansion;
- round 2: two contradicting papers on the lactylation → IFNG step make the chromatin route *Contradicted*;
- later rounds: exploration slots search the untested routes; GPR81 returns nothing twice → *Insufficient evidence: no study found after repeated searches*.

In [2]:
rt, scenario = offline_runtime()          # Settings overrides go here, e.g. offline_runtime(max_rounds=4)
final = run(scenario["question"], rt)

[parse] exposure=Lactate (LOCAL:lactate) outcome=CD8 T cell effector function (LOCAL:cd8_t_cell_effector_function) expected=down
[plan] round 1: 4 queries (coverage round)
[search] NEW_RESULTS: 4/4 queries ran, 9 new papers
[screen] 9 screened, 8 included, 1 retracted excluded
[extract] S001: 1 anchored, 0 dropped (abstract_only)
[extract] S002: 1 anchored, 0 dropped (abstract_only)
[extract] S003: 1 anchored, 0 dropped (abstract_only)
[extract] S004: 1 anchored, 0 dropped (abstract_only)
[extract] S005: 1 anchored, 0 dropped (abstract_only)
[extract] S006: 1 anchored, 0 dropped (abstract_only)
[extract] S007: 1 anchored, 0 dropped (abstract_only)
[extract] S009: 0 anchored, 1 dropped (abstract_only)
[normalize] 7 claims, 4 relations sent to LLM, 3 alias merges
[semantic] 2 candidate pairs (2 new) status=COMPLETE; 0 conflict candidates, 0 triaged
[portfolio] round 1: H1=Insufficient evidence:0.2, H2=Insufficient evidence:0.0, H3=Insufficient evidence:0.0, H4=Insufficient evidence:0.0 |

## §3. Inspect the run

**Pathway portfolio by round** — status and score of every pathway, and which links got the search budget.

In [3]:
pd.DataFrame([{"round": h["round"], "gate": h["gate"],
               **{hid: f"{st} {sc}" for hid, st, sc in h["hypotheses"]},
               "targets searched next": "; ".join(h["targets"])}
              for h in final["portfolio_history"]]).fillna("")

,round,gate,H1,H2,H3,H4,targets searched next,H5
0,1,TARGETED,Insufficient evidence 0.2,Insufficient evidence 0.0,Insufficient evidence 0.0,Insufficient evidence 0.0,Histone lactylation -> IFNG expression; IFNG expression -> CD8 T cell effect...,
1,2,TARGETED,Contradicted 0.067,Insufficient evidence 0.0,Insufficient evidence 0.0,Insufficient evidence 0.0,GPR81 signaling -> CD8 T cell effector function; Intracellular pH -> CD8 T c...,Contradicted 0.059
2,3,TARGETED,Contradicted 0.067,Insufficient evidence 0.0,Insufficient evidence 0.17,Insufficient evidence 0.0,Lactate -> Intracellular pH; NAD+ -> Glycolytic flux; GPR81 signaling -> CD8...,Contradicted 0.059
3,4,TARGETED,Contradicted 0.067,Insufficient evidence 0.0,Insufficient evidence 0.17,Supported 0.7,Lactate -> Intracellular pH; Intracellular pH -> CD8 T cell effector function,Contradicted 0.059
4,5,NO_TARGETS,Contradicted 0.067,Insufficient evidence 0.0,Insufficient evidence 0.17,Supported 0.7,,Contradicted 0.059


**Final pathways**

In [4]:
from bmira.portfolio import STATUS_LABEL, ORIGIN_LABEL, FLAG_LABEL
links = final["links"]
pd.DataFrame([{"id": h.id, "name": h.name, "verdict": STATUS_LABEL[h.status], "why": h.reason,
               "score": h.score, "source": ORIGIN_LABEL[h.origin],
               "logic warnings": "; ".join(FLAG_LABEL[f] for f in h.logic_flags),
               "path": " → ".join([links[h.links[0]].subject_label] + [links[k].object_label for k in h.links])}
              for h in final["hypotheses"]])

,id,name,verdict,why,score,source,logic warnings,path
0,H4,Redox-metabolic route,Supported,every step has independent support,0.700,LLM expansion,,Lactate → NAD+ → Glycolytic flux → CD8 T cell effector function
1,H3,Acidification route,Insufficient evidence,Lactate → Intracellular pH: no study found after repeated searches,0.170,LLM proposal,steps shown in different cell types,Lactate → Intracellular pH → CD8 T cell effector function
2,H1,Chromatin route,Contradicted,Histone lactylation → IFNG expression: 2 contradicting vs 1 supporting papers,0.067,LLM proposal,,Lactate → Histone lactylation → IFNG expression → CD8 T cell effector function
3,H5,"Literature-graph route via Histone lactylation, IFNG expression",Contradicted,Histone lactylation → IFNG expression: 2 contradicting vs 1 supporting papers,0.059,Found in literature graph,net direction conflicts with the question,Lactate → Histone lactylation → IFNG expression → CD8 T cell effector function
4,H2,Receptor route,Insufficient evidence,GPR81 signaling → CD8 T cell effector function: no study found after repeate...,0.000,LLM proposal,,Lactate → GPR81 signaling → CD8 T cell effector function


**Link evidence** (shared by every pathway that uses a link)

In [5]:
pd.DataFrame([{"step": f"{l.subject_label} -{l.relation}-> {l.object_label}",
               "verdict": STATUS_LABEL[l.status], "why": l.reason, "grade": l.grade,
               "papers": l.n_studies, "contra papers": l.n_contra_studies,
               "context-dependent": len(l.context_dependent_ids), "completeness": l.completeness,
               "times searched": l.times_targeted, "zero-yield": l.zero_yield_count}
              for l in links.values()]).sort_values("completeness", ascending=False)

,step,verdict,why,grade,papers,contra papers,context-dependent,completeness,times searched,zero-yield
7,NAD+ -increases-> Glycolytic flux,Supported,"2 papers, best grade strong",strong,2,0,0,1.000,1,0
1,Glycolytic flux -increases-> CD8 T cell effector function,Supported,"2 papers, best grade moderate",moderate,2,0,0,0.700,1,0
5,Lactate -decreases-> NAD+,Supported,"2 papers, best grade moderate",moderate,2,0,0,0.700,0,0
3,IFNG expression -increases-> CD8 T cell effector function,Supported,"2 papers, best grade moderate",moderate,2,0,0,0.700,1,0
10,Lactate -increases-> Histone lactylation,Supported,"2 papers, best grade moderate",moderate,2,0,0,0.700,0,0
2,Intracellular pH -increases-> CD8 T cell effector function,Insufficient evidence,1 of 2 required papers,moderate,1,0,0,0.350,2,1
11,Lactate -decreases-> Intracellular pH,Insufficient evidence,no study found after repeated searches,weak,1,0,0,0.200,2,2
0,Histone lactylation -increases-> IFNG expression,Contradicted,2 contradicting vs 1 supporting papers,moderate,1,2,0,0.117,0,0
8,Histone lactylation -no_effect-> IFNG expression,Contradicted,2 contradicting vs 1 supporting papers,moderate,1,2,0,0.117,0,0
9,Histone lactylation -decreases-> IFNG expression,Contradicted,2 contradicting vs 1 supporting papers,weak,1,2,0,0.067,1,0


**Claim ledger** and dropped claims

In [6]:
pd.DataFrame([{"id": c.id, "claim": f"{c.subject_label} -{c.relation_norm}-> {c.object_label}",
               "surface": c.relation_raw, "relation from": c.relation_source, "grade": c.grade,
               "limiting axis": c.grade_detail.get("limiting_axis"), "caps": c.grade_detail.get("caps"),
               "context": c.context_cell_type} for c in final["claims"]])

,id,claim,surface,relation from,grade,limiting axis,caps,context
0,CS001_0,Lactate -increases-> Histone lactylation,increased,lexical,moderate,design,[],CD8 T cell
1,CS002_0,Lactate -increases-> Histone lactylation,elevated,lexical,moderate,causal,[],CD8 T cell
2,CS003_0,Histone lactylation -decreases-> IFNG expression,repressed,llm,weak,design,[],CD8 T cell
3,CS004_0,IFNG expression -increases-> CD8 T cell effector function,supported,llm,moderate,design,[],CD8 T cell
4,CS005_0,Lactate -decreases-> NAD+,depleted,llm,moderate,design,[],CD8 T cell
5,CS006_0,Lactate -decreases-> NAD+,lowered,llm,moderate,causal,[],CD8 T cell
6,CS007_0,Lactate -decreases-> Intracellular pH,reduced,lexical,weak,design,[],tumor cell
7,CS012_0,Glycolytic flux -increases-> CD8 T cell effector function,enhanced,lexical,moderate,design,[],CD8 T cell
8,CS013_0,Glycolytic flux -increases-> CD8 T cell effector function,enhanced,lexical,moderate,causal,[],CD8 T cell
9,CS014_0,Histone lactylation -no_effect-> IFNG expression,did not change,llm,moderate,directness,[null_without_power],CD8 T cell


In [7]:
print("dropped (not anchored):", [(c.pmid, c.span[:70]) for c in final["dropped_claims"]])
print("excluded papers:", [(p.pmid, p.relevance_reason) for p in final["papers"] if p.screen_status == "excluded"])
print("LLM calls:", dict(rt.llm.calls))
print("LLM items:", dict(rt.llm.items))

dropped (not anchored): [('S009', 'Lactate abolished CD8 T cell effector function through histone lactyla')]
excluded papers: [('S008', 'retracted / expression of concern')]
LLM calls: {'parse': 1, 'entity': 11, 'plan': 5, 'screen': 4, 'extract': 16, 'relation': 3, 'alias': 1, 'pair': 3, 'seed': 1, 'expand': 1, 'conflict': 1, 'synthesize': 1, 'entailment': 1}
LLM items: {'parse': 1, 'entity': 11, 'plan': 5, 'screen': 16, 'extract': 16, 'relation': 6, 'alias': 3, 'pair': 8, 'seed': 1, 'expand': 1, 'conflict': 1, 'synthesize': 1, 'entailment': 14}


**Report** (synthesis + computed portfolio + run-quality warnings + verification)

In [8]:
from IPython.display import Markdown
Markdown(final["report"])

## Summary
The best-supported route is Redox-metabolic route [H4] [CS005_0] [CS006_0] [CS010_0] [CS011_0].

### [H4] Redox-metabolic route: Supported (score 0.7)
[L1] Lactate reduces NAD+ [CS005_0] [CS006_0].
[L2] NAD+ increases Glycolytic flux [CS010_0] [CS011_0].
[L3] Glycolytic flux increases CD8 T cell effector function [CS012_0] [CS013_0].

### [H3] Acidification route: Insufficient evidence (score 0.17)
[L4] Lactate is associated with changes in Intracellular pH [CS007_0].
[L5] Intracellular pH increases CD8 T cell effector function [CS016_0].

### [H1] Chromatin route: Contradicted (score 0.067)
[L6] Lactate increases Histone lactylation [CS001_0] [CS002_0].
[L7] Histone lactylation is associated with changes in IFNG expression [CS003_0].
[L7] Other studies report null or opposite findings for this step [CS014_0] [CS015_0].
[L8] IFNG expression increases CD8 T cell effector function [CS004_0] [CS017_0].

### [H5] Literature-graph route via Histone lactylation, IFNG expression: Contradicted (score 0.059)
[L6] Lactate increases Histone lactylation [CS001_0] [CS002_0].
[L9] Histone lactylation increases IFNG expression [CS015_0].
[L9] Other studies report null or opposite findings for this step [CS003_0] [CS014_0].
[L8] IFNG expression increases CD8 T cell effector function [CS004_0] [CS017_0].

### [H2] Receptor route: Insufficient evidence (score 0.0)
[L10] No study in the ledger examined the step from Lactate to GPR81 signaling [NO_EVIDENCE].
[L11] No study in the ledger examined the step from GPR81 signaling to CD8 T cell effector function [NO_EVIDENCE].

---
## Pathway portfolio (computed)
Search stopped after 5 rounds: nothing left to search. Scores rank pathways; they are not probabilities.

| Pathway | Verdict | Why | Score | Source | Logic warnings | Steps |
|---|---|---|---|---|---|---|
| H4 Redox-metabolic route | Supported | every step has independent support | 0.7 | LLM expansion | - | L1 Lactate→NAD+ (Supported, moderate); L2 NAD+→Glycolytic flux (Supported, strong); L3 Glycolytic flux→CD8 T cell effector function (Supported, moderate) |
| H3 Acidification route | Insufficient evidence | Lactate → Intracellular pH: no study found after repeated searches | 0.17 | LLM proposal | steps shown in different cell types | L4 Lactate→Intracellular pH (Insufficient evidence, weak); L5 Intracellular pH→CD8 T cell effector function (Insufficient evidence, moderate) |
| H1 Chromatin route | Contradicted | Histone lactylation → IFNG expression: 2 contradicting vs 1 supporting papers | 0.067 | LLM proposal | - | L6 Lactate→Histone lactylation (Supported, moderate); L7 Histone lactylation→IFNG expression (Contradicted); L8 IFNG expression→CD8 T cell effector function (Supported, moderate) |
| H5 Literature-graph route via Histone lactylation, IFNG expression | Contradicted | Histone lactylation → IFNG expression: 2 contradicting vs 1 supporting papers | 0.059 | Found in literature graph | net direction conflicts with the question | L6 Lactate→Histone lactylation (Supported, moderate); L9 Histone lactylation→IFNG expression (Contradicted); L8 IFNG expression→CD8 T cell effector function (Supported, moderate) |
| H2 Receptor route | Insufficient evidence | GPR81 signaling → CD8 T cell effector function: no study found after repeated searches | 0.0 | LLM proposal | - | L10 Lactate→GPR81 signaling (Insufficient evidence); L11 GPR81 signaling→CD8 T cell effector function (Insufficient evidence) |

## Run-quality warnings
- Candidate pairs came from the 'hashing' encoder, not sentence embeddings.

## Verification
- PASS

## §4. Tests
One test per exit gate of the development plan (P0–P5).

In [9]:
!python -m pytest -q tests

...........

..

.                                                           [100%]
14 passed in 1.30s


## §5. Live run

Needs an API key and network access to NCBI, Europe PMC and EBI OLS. Save the state afterwards so runs can be compared across versions.

In [10]:
RUN_LIVE = False
QUESTION = "Does lactate accumulation in the tumor microenvironment suppress CD8 T cell effector function, and through which mechanisms?"

if RUN_LIVE:
    import os
    from getpass import getpass
    settings = Settings(provider="openai", ncbi_email=os.environ.get("NCBI_EMAIL", ""),
                        ncbi_api_key=os.environ.get("NCBI_API_KEY", ""))
    key = os.environ.get("OPENAI_API_KEY") or getpass("OPENAI_API_KEY: ")
    live_rt = Runtime.live(settings, api_key=key)
    live = run(QUESTION, live_rt)
    display(Markdown(live["report"]))
    with open("run_state.json", "w") as f:
        json.dump({k: (v.model_dump() if hasattr(v, "model_dump") else v) for k, v in live.items()
                   if k in {"question", "hypotheses", "links", "portfolio_history", "warnings", "verification"}},
                  f, default=lambda o: o.model_dump() if hasattr(o, "model_dump") else str(o), indent=1)

## §6. Optional: review the seeded pathways before searching

With `interactive=True` the graph pauses once, right after seeding (the cheapest moment to correct the search). Resume with the pathways to drop.

In [11]:
from langgraph.types import Command
rt_i, sc = offline_runtime(interactive=True)
agent = build_agent(rt_i)
cfg = {"configurable": {"thread_id": "review-demo"}, "recursion_limit": 250}
paused = agent.invoke({"question": sc["question"]}, cfg)
print(paused["__interrupt__"][0].value)
resumed = agent.invoke(Command(resume={"drop": ["H2"]}), cfg)
print([(h.id, h.name, STATUS_LABEL[h.status]) for h in resumed["hypotheses"]])

[parse] exposure=Lactate (LOCAL:lactate) outcome=CD8 T cell effector function (LOCAL:cd8_t_cell_effector_function) expected=down
[plan] round 1: 4 queries (coverage round)
[search] NEW_RESULTS: 4/4 queries ran, 9 new papers
[screen] 9 screened, 8 included, 1 retracted excluded


Deserializing unregistered type bmira.schemas.ParsedQuestion from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('bmira.schemas', 'ParsedQuestion')]


Deserializing unregistered type bmira.schemas.SearchQuery from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('bmira.schemas', 'SearchQuery')]


Deserializing unregistered type bmira.schemas.Paper from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('bmira.schemas', 'Paper')]


Deserializing unregistered type bmira.schemas.Claim from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('bmira.schemas', 'Claim')]


Deserializing unregistered type bmira.schemas.PairAdjudication from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('bmira.schemas', 'PairAdjudication')]


Deserializing unregistered type bmira.schemas.LinkEvidence from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('bmira.schemas', 'LinkEvidence')]


Deserializing unregistered type bmira.schemas.Hypothesis from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('bmira.schemas', 'Hypothesis')]


[extract] S001: 1 anchored, 0 dropped (abstract_only)
[extract] S002: 1 anchored, 0 dropped (abstract_only)
[extract] S003: 1 anchored, 0 dropped (abstract_only)
[extract] S004: 1 anchored, 0 dropped (abstract_only)
[extract] S005: 1 anchored, 0 dropped (abstract_only)
[extract] S006: 1 anchored, 0 dropped (abstract_only)
[extract] S007: 1 anchored, 0 dropped (abstract_only)
[extract] S009: 0 anchored, 1 dropped (abstract_only)
[normalize] 7 claims, 4 relations sent to LLM, 3 alias merges
[semantic] 2 candidate pairs (2 new) status=COMPLETE; 0 conflict candidates, 0 triaged
[portfolio] round 1: H1=Insufficient evidence:0.2, H2=Insufficient evidence:0.0, H3=Insufficient evidence:0.0, H4=Insufficient evidence:0.0 | searching | targets=['Histone lactylation->IFNG expression', 'IFNG expression->CD8 T cell effector function', 'Glycolytic flux->CD8 T cell effector function']
{'message': "Drop pathways by id, e.g. {'drop': ['H2']}", 'pathways': [('H1', 'Chromatin route', 'Insufficient evidenc

[normalize] 12 claims, 1 relations sent to LLM, 0 alias merges
[semantic] 7 candidate pairs (5 new) status=COMPLETE; 1 conflict candidates, 1 triaged
[portfolio] round 2: H1=Contradicted:0.067, H5=Contradicted:0.059, H3=Insufficient evidence:0.0, H4=Insufficient evidence:0.0 | searching | targets=['Intracellular pH->CD8 T cell effector function', 'NAD+->Glycolytic flux', 'Lactate->Intracellular pH']
[plan] round 3: 9 queries for 3 target links
[search] NEW_RESULTS: 9/9 queries ran, 3 new papers
[screen] 3 screened, 3 included, 0 retracted excluded
[extract] S010: 1 anchored, 0 dropped (abstract_only)
[extract] S011: 1 anchored, 0 dropped (full_text)
[extract] S016: 1 anchored, 0 dropped (abstract_only)
[normalize] 15 claims, 1 relations sent to LLM, 0 alias merges
[semantic] 8 candidate pairs (1 new) status=COMPLETE; 1 conflict candidates, 1 triaged
[portfolio] round 3: H4=Supported:0.7, H3=Insufficient evidence:0.17, H1=Contradicted:0.067, H5=Contradicted:0.059 | searching | targets=[